## 1. Taxpayers — Bronze Layer

Load the raw `taxpayers.csv` from the Bronze layer as-is, with no transformations.
This preserves the original messy data for traceability — every cleaning
decision below can be traced back to this untouched source.


In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
df_taxpayers= spark.read.option("header",True)\
.csv("Files/bronze/taxpayers.csv")

df_taxpayers.printSchema()
display(df_taxpayers.limit(10))

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 3, Finished, Available, Finished, False)

root
 |-- taxpayer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- vat_number: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- annual_revenue_sar: string (nullable = true)
 |-- company_size: string (nullable = true)
 |-- city: string (nullable = true)
 |-- is_registered: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 400763c9-fc72-49b7-951e-935b44fad174)

### 1.1 Duplicate Detection

Before cleaning any column, check whether the same `taxpayer_id` appears more
than once. A generic `dropDuplicates()` (exact row match) did not catch these,
which means the duplicate rows differ slightly in at least one column —
so duplicates need to be identified by business key (`taxpayer_id`), not by
full-row equality.


In [2]:
%%sql
select taxpayer_id,count(*)
from taxpayers
group by taxpayer_id
HAVING count(*)>1;

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 4, Finished, Available, Finished, False)

<Spark SQL result set with 16 rows and 2 fields>

### 1.2 Deduplication Rule

Business rule: when the same `taxpayer_id` appears more than once, keep the
row with the most recent `registration_date`. This assumes the latest record
is the most trustworthy, rather than dropping duplicates arbitrarily.

In [3]:
%%sql 
create or replace temp view taxpayer_dedupl as
select * from(
select *,row_number() over (PARTITION by taxpayer_id order by registration_date)row_num 
from taxpayers) ranked
where row_num = 1;

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [4]:
%%sql 
select count(*) from taxpayer_dedupl;

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 6, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 1.3 Standardizing Taxpayer Attributes

Cleaning applied in one pass:
- `sector`: typos and casing variants mapped to 10 standard categories
- `city`: casing normalized, and known transliteration variants
  (Mecca/Makkah, Medina/Madinah) merged into one spelling
- `is_registered`: Yes/Y/1/True and No/N/0/False variants standardized to
  a single boolean-style label; anything unrecognized is left NULL rather
  than guessed
- `registration_date`: parsed across 5 known date formats found in the raw
  data using COALESCE, so each row is converted using whichever format
  actually matches it
- `annual_revenue_sar`: comma-formatted strings converted to numbers,
  and erroneous negative values corrected with ABS() (revenue cannot be
  negative)
- `vat_number`: flagged as Valid / Malformed / Missing rather than dropped
  or corrected — a VAT number is compliance-critical, so guessing a value
  is worse than leaving it visibly flagged
- `has_registration_date`: a `has_registration_date` flag is
added so completeness can be measured explicitly (e.g. "X% of taxpayer
records are missing a registration date") rather than the missingness
being silently invisible.


In [5]:
%%sql
CREATE OR REPLACE TEMP VIEW taxpayer_clean AS
 SELECT taxpayer_id, company_name, vat_number, 
   CASE
    WHEN LOWER(TRIM(sector)) IN ('retail','retial') THEN 'Retail'
    WHEN LOWER(TRIM(sector)) IN ('construction','contsruction') THEN 'Construction'
    WHEN LOWER(TRIM(sector)) IN ('f&b','f & b','food & beverage','fnb') THEN 'F&B'
    WHEN LOWER(TRIM(sector)) IN ('healthcare','health care') THEN 'Healthcare'
    WHEN LOWER(TRIM(sector)) IN ('education','educaton') THEN 'Education'
    WHEN LOWER(TRIM(sector)) IN ('manufacturing','manufactring') THEN 'Manufacturing'
    WHEN LOWER(TRIM(sector)) IN ('real estate','realestate') THEN 'Real Estate'
    WHEN LOWER(TRIM(sector)) IN ('technology','tech') THEN 'Technology'
    WHEN LOWER(TRIM(sector)) IN ('logistics','logisitcs') THEN 'Logistics'
    ELSE 'Other'
  END AS sector,
  
  COALESCE(
     TO_DATE(registration_date, 'yyyy-MM-dd'), 
      TO_DATE(registration_date, 'dd/MM/yyyy'), 
      TO_DATE(registration_date, 'MM-dd-yyyy'), 
      TO_DATE(registration_date, 'dd-MMM-yyyy'),
      TO_DATE(registration_date, 'yyyy/MM/dd')) AS registration_date,

abs(round(cast(replace(cast(annual_revenue_sar as String),',','')as DOUBLE),0))
     as annual_revenue_sar, company_size, 
  case 
     when lower(trim(city)) in ('jedah','jeddah')then 'Jeddah'
     when lower(trim(city)) in ('khobar','al khobar')then 'Khobar'
     when lower(trim(city)) in ('damam','dammam')then  'Dammam'
     when lower(trim(city)) in('madinah','medina') then 'Medina'
     when lower(trim(city)) in ('riyad','riyadh') then 'Riyadh'
     when lower(trim(city)) in ('mecca','makkah') then 'Makkah'
     else TRIM(INITCAP(city)) 
 end AS city,
 case
    when lower(trim(is_registered))  in('N','no','false','0')then 'No'
    when lower(trim(is_registered)) in ('y','yes','1')then 'Yes'
    else Null
end as is_registered,
case
     when length(vat_number)is null then "Missing"
     when length(vat_number)==17 then "Valid"
     else "Milformed"
 end as vat_number_status,
 CASE WHEN registration_date IS NULL THEN 0 ELSE 1 END AS has_registration_date
  
  FROM taxpayer_dedupl

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [6]:
%%sql
select * from taxpayer_clean


StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 8, Finished, Available, Finished, False)

<Spark SQL result set with 800 rows and 11 fields>

### 1.4 Persisting to Silver Layer

Write the cleaned view as a permanent Delta table. Temp views only exist
for the notebook session; this makes the cleaned data durable and queryable
by other notebooks/reports.

In [7]:
%%sql
create or replace table silver_taxpayers as
select * from taxpayer_clean

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [8]:
%%sql
SELECT COUNT(*) FROM silver_taxpayers

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 10, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

## 2. Invoices — Bronze Layer

Same pattern as Taxpayers: load raw, unmodified data first.

In [9]:
%%sql
select * from invoices

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 11, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 10 fields>

### 2.1 Duplicate Removal

Exact duplicate rows by `invoice_id` are removed, keeping the most recent
`invoice_date` per duplicate group.

In [10]:
%%sql
create or replace temp  view invoice_dedup as
select *from(
   select *,
     row_number() 
     over(partition by invoice_id order by taxpayer_id desc )as ranked
      from  invoices)
where ranked=1


StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 12, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 2.2 Referential Integrity Check

Some invoices reference a `taxpayer_id` that does not exist in the cleaned
`taxpayers` table ("orphan" records) — simulating a real-world data quality
issue (e.g. system desync, deleted taxpayer record still referenced
elsewhere). Verified using two independent SQL techniques (LEFT JOIN and
EXCEPT) to cross-check the result.



In [11]:
%%sql
select distinct i.taxpayer_id from invoice_dedup i
left join silver_taxpayers t
on i.taxpayer_id = t.taxpayer_id
where t.taxpayer_id is null

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 13, Finished, Available, Finished, False)

<Spark SQL result set with 81 rows and 1 fields>

In [12]:
%%sql
select count(*) from invoice_dedup

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 14, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

### 2.3 Standardizing Invoice Attributes

- `invoice_date`: same multi-format COALESCE parsing as taxpayers
- `invoice_type`, `customer_type`, `qr_code_present`: typo/casing variants
  standardized
- `amount_before_vat`: currency text (e.g. "SAR 1,200") and comma
  separators stripped, converted to numeric
- `vat_anomaly_flag`: flags any invoice where the recorded `vat_amount`
  differs from the expected 15% of `amount_before_vat` by more than
  0.5 SAR. This is the core feature that will drive anomaly detection
  later in the project.

In [13]:
%%sql
CREATE OR REPLACE TEMP VIEW invoices_clean AS
SELECT invoice_id,taxpayer_id,
COALESCE(
      TO_DATE(invoice_date, 'yyyy-MM-dd'), 
      TO_DATE(invoice_date, 'dd/MM/yyyy'), 
      TO_DATE(invoice_date, 'MM-dd-yyyy'), 
      TO_DATE(invoice_date, 'dd-MMM-yyyy'),
      TO_DATE(invoice_date, 'yyyy/MM/dd')) AS invoice_date,
case 
     when trim(invoice_type) in ('Simplifed','Simplified') then 'Simplified'
     when lower(trim(invoice_type)) in ('standard') then 'Standard'
end as invoice_type,

cast(round(replace(trim( REPLACE(amount_before_vat, 'SAR', '')),',',''),1)as double) as
 amount_before_vat,round(vat_amount,1)as vat_amount
,round(total_amount,1)as total_amount,
case when lower(trim(customer_type))in ('b2b') then 'B2B'
     when lower(trim(customer_type))in ('b2c') then 'B2C' end as customer_type,
case when qr_code_present in ('False','N','0','No','no','0')then 'False'
     when qr_code_present in ('Y','Yes','yes','True','1')then 'True'
     else null
     end as qr_code_present,
            status,
CASE WHEN ABS(vat_amount - (amount_before_vat * 0.15)) > 0.5 THEN 1 ELSE 0 END AS vat_anomaly_flag
FROM invoice_dedup 

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [14]:
%%sql
select * from invoices_clean

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 16, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 11 fields>

### 2.4 Inspecting VAT Anomalies

Exploratory check of the flagged rows — 240 invoices (~4% of the table)
show a VAT discrepancy larger than 0.5 SAR, consistent with the ~5%
anomaly rate intentionally built into the synthetic dataset.

In [15]:
%%sql
select invoice_id,amount_before_vat,vat_amount,
round((amount_before_vat*.15),2)expexted_value,
round(vat_amount-((amount_before_vat*.15)),2) as difference
from invoices_clean
where round(vat_amount-((amount_before_vat*.15)),2) > 0.5
order by 1 desc

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 17, Finished, Available, Finished, False)

<Spark SQL result set with 94 rows and 5 fields>

In [16]:
%%sql
SELECT vat_anomaly_flag, COUNT(*) FROM invoices_clean GROUP BY vat_anomaly_flag

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 18, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 2 fields>

### 2.5 Persisting to Silver Layer

In [17]:
%%sql 
create or replace table silver_invoices as
select * from invoices_clean

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 19, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## 3. VAT Declarations — Bronze Layer

In [18]:
%%sql
select * from vat_declarations

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 20, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 10 fields>

## 3.1 Check duplicate

In [19]:
%%sql
SELECT declaration_id, COUNT(*) AS cnt
FROM vat_declarations
GROUP BY declaration_id
HAVING COUNT(*) > 1

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

### 3.2 Standardizing Declaration Attributes

- `period_type`: derived from the raw `period` value — Saudi VAT
  declarations can be filed monthly or quarterly depending on business
  size, so both formats are legitimate and are kept, tagged by type
  rather than forced into one format
- `due_date`, `submission_date`: multi-format date parsing
- `declared_sales`, `declared_vat`, `paid_amount`: rounded to 2 decimals
- `outstanding_amount`: calculated as `declared_vat - paid_amount`,
  representing real outstanding tax arrears per declaration —
  a key compliance metric for the Gold layer

In [20]:
%%sql
CREATE OR REPLACE TEMP VIEW declarations_clean AS
SELECT
    declaration_id,
    taxpayer_id,
    period,
    CASE WHEN period LIKE '%Q%' THEN 'Quarterly' ELSE 'Monthly' END AS period_type,
    COALESCE(
        TO_DATE(due_date, 'yyyy-MM-dd'),
        TO_DATE(due_date, 'dd/MM/yyyy'),
        TO_DATE(due_date, 'MM-dd-yyyy'),
        TO_DATE(due_date, 'dd-MMM-yyyy'),
        TO_DATE(due_date, 'yyyy/MM/dd')
    ) AS due_date,
    COALESCE(
        TO_DATE(submission_date, 'yyyy-MM-dd'),
        TO_DATE(submission_date, 'dd/MM/yyyy'),
        TO_DATE(submission_date, 'MM-dd-yyyy'),
        TO_DATE(submission_date, 'dd-MMM-yyyy'),
        TO_DATE(submission_date, 'yyyy/MM/dd')
    ) AS submission_date,
    ROUND(declared_sales, 2) AS declared_sales,
    ROUND(declared_vat, 2) AS declared_vat,
    ROUND(paid_amount, 2) AS paid_amount,
    ROUND(declared_vat - paid_amount, 2) AS outstanding_amount,
    status

FROM vat_declarations

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 22, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [21]:
%%sql
select * from declarations_clean

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 23, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 11 fields>

### 3.3 Persisting to Silver Layer

In [22]:
%%sql
CREATE OR REPLACE TABLE silver_declarations AS
SELECT * FROM declarations_clean

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## 4. Violations — Bronze Layer

In [23]:
%%sql 
select * from violations

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 25, Finished, Available, Finished, False)

<Spark SQL result set with 650 rows and 7 fields>

## 4.1 Check duplicate

In [24]:

%%sql
--check duplicate
select violation_id ,count(*)as cnt
from violations
group by violation_id
having count(*) >1

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 26, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 2 fields>

### 4.2 Standardizing Violation Attributes

- `violation_type`: 5 categories standardized from typo/casing variants
  using pattern matching
- `violation_date`: multi-format date parsing
- `penalty_amount_sar`: currency text and commas stripped; erroneous
  negative penalties corrected with ABS()

In [25]:
%%sql
CREATE OR REPLACE TEMP VIEW violations_clean AS
SELECT
    violation_id,
    taxpayer_id,
    CASE
        WHEN LOWER(TRIM(violation_type)) LIKE '%late registration%' THEN 'Late Registration'
        WHEN LOWER(TRIM(violation_type)) LIKE '%late filing%' THEN 'Late Filing'
        WHEN LOWER(TRIM(violation_type)) LIKE '%e-invoicing%' THEN 'E-invoicing Non-compliance'
        WHEN LOWER(TRIM(violation_type)) LIKE '%underreporting%' THEN 'Underreporting'
        WHEN LOWER(TRIM(violation_type)) LIKE '%qr%' THEN 'Missing QR Code'
        ELSE violation_type
    END AS violation_type,
    COALESCE(
        TO_DATE(violation_date, 'yyyy-MM-dd'),
        TO_DATE(violation_date, 'dd/MM/yyyy'),
        TO_DATE(violation_date, 'MM-dd-yyyy'),
        TO_DATE(violation_date, 'dd-MMM-yyyy'),
        TO_DATE(violation_date, 'yyyy/MM/dd')
    ) AS violation_date,
    ROUND(ABS(CAST(REPLACE(REPLACE(CAST(penalty_amount_sar AS STRING), 'SAR', ''), ',', '') AS DOUBLE)), 2) AS penalty_amount_sar,
    payment_status,
    COALESCE(
        TO_DATE(resolution_date, 'yyyy-MM-dd'),
        TO_DATE(resolution_date, 'dd/MM/yyyy'),
        TO_DATE(resolution_date, 'MM-dd-yyyy'),
        TO_DATE(resolution_date, 'dd-MMM-yyyy'),
        TO_DATE(resolution_date, 'yyyy/MM/dd')
    ) AS resolution_date
FROM violations

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 27, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### 4.3 Persisting to Silver Layer

All four Silver tables are now complete: `silver_taxpayers`,
`silver_invoices`, `silver_declarations`, `silver_violations`. This closes
the data cleaning phase (Bronze → Silver) of the medallion architecture.
Next: Gold layer business-logic tables built on top of these.

In [26]:
%%sql
CREATE OR REPLACE TABLE silver_violations AS
SELECT * FROM violations_clean

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 28, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [27]:
%%sql 
select * from silver_violations

StatementMeta(, 37532f3f-7a03-47f6-9e7e-c3c160605e30, 29, Finished, Available, Finished, False)

<Spark SQL result set with 650 rows and 7 fields>